<a href="https://colab.research.google.com/github/saffarizadeh/LLMs/blob/main/Using_LLMs_via_API_(A).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://kambizsaffari.com/Logo/College_of_Business.cmyk-hz-lg.png" width="500px"/>

# *Artificial Intelligence for Business*

# **A Quick Introduction to Using Major LLMs via API (Part A)**

Instructor: Dr. Kambiz Saffari

---

## What we will do today (≈ 60 min)

In this session, we will learn how to **send text to an LLM over the internet and get an answer back** - in code. This is how apps talk to ChatGPT, Claude, and Gemini behind the scenes.

We will use a small dataset of financial tweets and ask three different LLMs to score the sentiment of each tweet.


> **No prior programming needed.** Just run each cell in order (`Shift + Enter`) and read the explanations.

---
## 1. Setup and API keys (*~5 min*)

An **API key** is like a password that tells the LLM provider "this request is from me, bill it to my account." You get one key per provider by signing up on their website.

Paste the three keys I shared into the cell below. **Keep them private** - never commit them to GitHub or share them publicly.

In [1]:
openai_api_key = ''  # from https://platform.openai.com/
claude_api_key = ''  # from https://console.anthropic.com/
gemini_api_key = ''  # from https://aistudio.google.com/

In [2]:
import numpy as np
import pandas as pd

---
## 2. Load our data (*~3 min*)

We have a small Excel file with 10 financial tweets. Upload `sentiment_10.xlsx` to Colab (folder icon on the left → upload) before running the next cell.

In [3]:
df_original = pd.read_excel('sentiment_10.xlsx')

In [4]:
df = df_original.copy()  # work on a copy so we can reset easily
df.head()

,Sentence,Sentiment
0,The GeoSolutions technology will leverage Bene...,positive
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,According to the Finnish-Russian Chamber of Co...,neutral
4,The Swedish buyout firm has sold its remaining...,neutral


---
## 3. Design a prompt (*~5 min*)

A **prompt** is the instruction you give the LLM. The better the prompt, the better the answer. Good prompts typically include:

- A **role** ("You are an expert financial analyst…")
- The **task** ("…score this tweet from –1.0 to 1.0")
- The **output format** ("…return only a number, nothing else")

We will use the same prompt for all three providers so the comparison is fair.

In [5]:
prompt = '''
You are an expert financial sentiment analyst.
Input: a tweet that may contain a target stock ticker.
Task: return a single numeric score between -1.0 (extremely negative)
      and 1.0 (extremely positive) reflecting the financial sentiment.
Output: ONLY the number. No words, no punctuation, no explanation.
'''

---
## 4. OpenAI GPT (*~15 min*)

Docs: [Platform overview](https://platform.openai.com/docs/overview) · [Pricing](https://platform.openai.com/docs/pricing) · [Chat API](https://platform.openai.com/docs/api-reference/responses)

In [6]:
from openai import OpenAI
openai_client = OpenAI(api_key=openai_api_key)

### 4a. A first "hello world" call

Every API call follows the same pattern: you pick a **model**, give it **instructions** (the role), give it **input** (the user message), and you get a response back.

In [7]:
response = openai_client.responses.create(
    model = 'gpt-4.1',
    instructions = 'You are a helpful assistant.',
    input = "Hey! What's up?!",
    temperature = 1
)

In [8]:
response  # the full response object - lots of metadata

Response(id='resp_00d2bd8444baa0280069e7ee72bc608197a16c8aa7625dbd4c', created_at=1776807538.0, error=None, incomplete_details=None, instructions='You are a helpful assistant.', metadata={}, model='gpt-4.1-2025-04-14', object='response', output=[ResponseOutputMessage(id='msg_00d2bd8444baa0280069e7ee7304d48197b0c0b1e5eac57ab2', content=[ResponseOutputText(annotations=[], text='Hey! Not much, just here and ready to help you out. What’s on your mind today?', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=1776807539.0, conversation=None, max_output_tokens=None, max_tool_calls=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_retention='in_memory', reasoning=Reasoning(effort=None, generate_summary=None, summary=None), safety_identifier=None, service_tier='default', status='completed', text=R

In [9]:
response.output_text  # just the text the model wrote

'Hey! Not much, just here and ready to help you out. What’s on your mind today?'

**About `temperature`:** it controls how random the output is. `0` = always the same answer. `2` = very creative / chaotic. For classification tasks like ours, low temperature (0.0–0.3) is best.

### 4b. Apply GPT to our dataset

Now we wrap the call in a function and apply it to every tweet.

In [10]:
def evaluate_content_gpt(prompt, content):
    full_prompt = f'{prompt}\n\nTweet:\n{content}'
    response = openai_client.responses.create(
        model = 'gpt-4.1',
        input = full_prompt,
        temperature = 0
    )
    return response.output_text.strip()

In [11]:
df['gpt_sentiment'] = df.Sentence.apply(lambda x: evaluate_content_gpt(prompt, x))

In [12]:
df['gpt_sentiment'] = df.gpt_sentiment.astype(float)  # text -> number

In [13]:
df

,Sentence,Sentiment,gpt_sentiment
0,The GeoSolutions technology will leverage Bene...,positive,0.8
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative,-0.9
2,"For the last quarter of 2010 , Componenta 's n...",positive,0.7
3,According to the Finnish-Russian Chamber of Co...,neutral,0.0
4,The Swedish buyout firm has sold its remaining...,neutral,0.0
5,$SPY wouldn't be surprised to see a green close,positive,0.4
6,Shell's $70 Billion BG Deal Meets Shareholder ...,negative,-0.4
7,SSH COMMUNICATIONS SECURITY CORP STOCK EXCHANG...,negative,-0.8
8,Kone 's net sales rose by some 14 % year-on-ye...,positive,0.8
9,The Stockmann department store will have a tot...,neutral,0.3


---
## 5. Anthropic Claude (*~12 min*)

Docs: [Console](https://console.anthropic.com/) · [Pricing](https://www.anthropic.com/pricing#api) · [Messages API](https://docs.anthropic.com/en/api/messages)

The same task, a different company, a slightly different API. Notice how the ideas are identical: pick a model, give it instructions, get a response.

In [14]:
!pip install anthropic --quiet

In [15]:
import anthropic
claude_client = anthropic.Anthropic(api_key=claude_api_key)

### 5a. A first "hello world" call

In [16]:
response = claude_client.messages.create(
    model = 'claude-sonnet-4-20250514',
    max_tokens = 1024,
    system = 'You are a helpful assistant.',
    messages = [{'role': 'user', 'content': "Hey! What's up?!"}]
)

/tmp/ipykernel_4539/3414069204.py:1: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = claude_client.messages.create(


In [17]:
response.content[0].text  # Claude's reply

"Hey there! Not much happening on my end - just here and ready to chat! How's your day going? What's got you in such an upbeat mood? 😊"

### 5b. Apply Claude to our dataset

In [18]:
def evaluate_content_claude(prompt, content):
    full_prompt = f'{prompt}\n\nTweet:\n{content}'
    response = claude_client.messages.create(
        model = 'claude-sonnet-4-20250514',
        max_tokens = 1024,
        messages = [{'role': 'user', 'content': full_prompt}]
    )
    return response.content[0].text.strip()

In [19]:
df['claude_sentiment'] = df.Sentence.apply(lambda x: evaluate_content_claude(prompt, x))

/tmp/ipykernel_4539/2117635143.py:3: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  response = claude_client.messages.create(


In [20]:
df['claude_sentiment'] = df.claude_sentiment.astype(float)

In [21]:
df

,Sentence,Sentiment,gpt_sentiment,claude_sentiment
0,The GeoSolutions technology will leverage Bene...,positive,0.8,0.6
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative,-0.9,-0.8
2,"For the last quarter of 2010 , Componenta 's n...",positive,0.7,0.7
3,According to the Finnish-Russian Chamber of Co...,neutral,0.0,0.2
4,The Swedish buyout firm has sold its remaining...,neutral,0.0,0.1
5,$SPY wouldn't be surprised to see a green close,positive,0.4,0.3
6,Shell's $70 Billion BG Deal Meets Shareholder ...,negative,-0.4,-0.4
7,SSH COMMUNICATIONS SECURITY CORP STOCK EXCHANG...,negative,-0.8,-0.7
8,Kone 's net sales rose by some 14 % year-on-ye...,positive,0.8,0.7
9,The Stockmann department store will have a tot...,neutral,0.3,0.2


---
## 6. Google Gemini (*~12 min*)

Docs: [Gemini API](https://ai.google.dev/gemini-api/docs) · [Text generation](https://ai.google.dev/gemini-api/docs/text-generation)

In [22]:
!pip install google-genai --quiet

In [23]:
from google import genai
gemini_client = genai.Client(api_key=gemini_api_key)

### 6a. A first "hello world" call

In [24]:
response = gemini_client.models.generate_content(
    model = 'gemini-3.1-pro-preview',
    contents = "Hey! What's up?!",
    config = genai.types.GenerateContentConfig(
        system_instruction = 'You are a helpful assistant.'
    )
)

In [25]:
response.text

"Hey there! Not much, just hanging out in the digital world and ready to help you out. How are you doing today? What's on your mind?"

### 6b. Apply Gemini to our dataset

In [26]:
def evaluate_content_gemini(prompt, content):
    full_prompt = f'{prompt}\n\nTweet:\n{content}'
    response = gemini_client.models.generate_content(
        model = 'gemini-3.1-pro-preview',
        contents = full_prompt,
        config = genai.types.GenerateContentConfig(temperature=0)
    )
    return response.text.strip()

In [27]:
df['gemini_sentiment'] = df.Sentence.apply(lambda x: evaluate_content_gemini(prompt, x))

In [28]:
df['gemini_sentiment'] = df.gemini_sentiment.astype(float)

In [29]:
df

,Sentence,Sentiment,gpt_sentiment,claude_sentiment,gemini_sentiment
0,The GeoSolutions technology will leverage Bene...,positive,0.8,0.6,0.75
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative,-0.9,-0.8,-1.00
2,"For the last quarter of 2010 , Componenta 's n...",positive,0.7,0.7,0.85
3,According to the Finnish-Russian Chamber of Co...,neutral,0.0,0.2,0.00
4,The Swedish buyout firm has sold its remaining...,neutral,0.0,0.1,0.00
5,$SPY wouldn't be surprised to see a green close,positive,0.4,0.3,0.50
6,Shell's $70 Billion BG Deal Meets Shareholder ...,negative,-0.4,-0.4,-0.60
7,SSH COMMUNICATIONS SECURITY CORP STOCK EXCHANG...,negative,-0.8,-0.7,-0.85
8,Kone 's net sales rose by some 14 % year-on-ye...,positive,0.8,0.7,0.80
9,The Stockmann department store will have a tot...,neutral,0.3,0.2,0.50


---
## 7. Compare and save (*~5 min*)

Let's look at how the three models agree (or disagree) with each other.

In [30]:
df[['gpt_sentiment', 'claude_sentiment', 'gemini_sentiment']].corr()

,gpt_sentiment,claude_sentiment,gemini_sentiment
gpt_sentiment,1.000000,0.986148,0.989924
claude_sentiment,0.986148,1.000000,0.980563
gemini_sentiment,0.989924,0.980563,1.000000


Save the results to an Excel file you can download.

In [31]:
df.to_excel('Results.xlsx', index=False)

**Discussion questions:**

1. Which tweets did the models disagree on the most? Why?
2. If you were building a real product, which model would you choose, and why? (Hint: think about cost, speed, and accuracy.)
3. What would happen if you changed the prompt? Try it in the next section.

---
# Try at home (extras)

These sections are not part of the 60-minute class. Run them on your own to go deeper.

## Extra 1 - Reasoning models

OpenAI's **reasoning models** (like `gpt-5`, `o3`) think step-by-step internally before answering. They are slower and more expensive, but much better at hard problems (math, planning, complex analysis).

Docs: [Reasoning](https://platform.openai.com/docs/guides/reasoning)

In [32]:
def evaluate_content_reasoning(prompt, content):
    full_prompt = f'{prompt}\n\nTweet:\n{content}'
    response = openai_client.responses.create(
        model = 'gpt-5.4',
        input = full_prompt,
        reasoning = {'effort': 'low'}  # low, medium, or high
    )
    return response.output_text.strip()

# Warning: this is ~10x slower and more expensive than gpt-4.1.
df['gpt_reasoning_sentiment'] = df.Sentence.apply(lambda x: evaluate_content_reasoning(prompt, x))
df['gpt_reasoning_sentiment'] = df.gpt_reasoning_sentiment.astype(float)
df

,Sentence,Sentiment,gpt_sentiment,claude_sentiment,gemini_sentiment,gpt_reasoning_sentiment
0,The GeoSolutions technology will leverage Bene...,positive,0.8,0.6,0.75,0.72
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative,-0.9,-0.8,-1.00,-0.95
2,"For the last quarter of 2010 , Componenta 's n...",positive,0.7,0.7,0.85,0.85
3,According to the Finnish-Russian Chamber of Co...,neutral,0.0,0.2,0.00,0.10
4,The Swedish buyout firm has sold its remaining...,neutral,0.0,0.1,0.00,-0.10
5,$SPY wouldn't be surprised to see a green close,positive,0.4,0.3,0.50,0.40
6,Shell's $70 Billion BG Deal Meets Shareholder ...,negative,-0.4,-0.4,-0.60,-0.50
7,SSH COMMUNICATIONS SECURITY CORP STOCK EXCHANG...,negative,-0.8,-0.7,-0.85,-0.80
8,Kone 's net sales rose by some 14 % year-on-ye...,positive,0.8,0.7,0.80,0.67
9,The Stockmann department store will have a tot...,neutral,0.3,0.2,0.50,0.40


## Extra 2 - The temperature knob

Run the same question at different temperatures and see how the style of the answer changes.

In [33]:
for t in [0, 0.5, 1.0, 1.5, 2.0]:
    r = openai_client.responses.create(
        model='gpt-4.1',
        input='Write a one-sentence slogan for a new coffee shop.',
        temperature=t
    )
    print(f'temp={t}:', r.output_text)

temp=0: Sip, Savor, Smile—Your Perfect Brew Awaits!
temp=0.5: Sip, Savor, Smile—Your Perfect Brew Awaits!
temp=1.0: Wake Up to Something Wonderful—Where Every Cup is a Fresh Start!
temp=1.5: Sip, Savor, Smile—Your Perfect Brew Awaits!
temp=2.0: Brewing Community, One Cup at a Time.


## Extra 3 - Try a different prompt

Edit the prompt below and re-run Section 4b. How do the results change if you:

- Ask for a score from 1 to 10 instead of –1 to 1?
- Ask the model to also return the stock ticker it saw?
- Remove the "expert" role?
- Add an example (one-shot prompting)?